# CodeBuddy — Fine-tuning Gemma 4 dengan Unsloth

Notebook ini melakukan fine-tuning Gemma 4 menggunakan dataset tutoring Bahasa Indonesia CodeBuddy.

**Runtime yang dibutuhkan:** GPU T4 (gratis di Colab) atau lebih baik

**Estimasi waktu:** beberapa jam (tergantung GPU Colab)

**Catatan VRAM (Colab T4 ~15 GB):** Step 2 & Step 6 memakai `max_seq_length=1024`, batch kecil, dan `gradient_checkpointing` agar tidak OOM. Jika masih OOM, di Step 6 turunkan `MAX_LEN` ke `768` atau `num_train_epochs` ke `2`, lalu **Runtime → Restart** dan jalankan ulang dari Step 1.

**Target:** Unsloth Prize (Google Gemma 4 Good Hackathon)

In [ ]:
# Step 1: Install Unsloth
!pip install unsloth -q
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo -q

In [ ]:
# Step 2: Load Gemma 4 dengan Unsloth
# max_seq_length 1024: aman untuk Colab T4 (~15 GB VRAM) bersama LoRA + training
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-4-e2b-it-unsloth-bnb-4bit',  # E2B fit di T4
    max_seq_length=1024,
    load_in_4bit=True,
)

print('Model loaded:', model.config.model_type)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 3: Tambahkan LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=42,
)

In [ ]:
# Step 4: Load dan convert dataset CodeBuddy
import json
from datasets import Dataset

# Upload file ini ke Colab atau load dari Google Drive
# File: backend/data/finetune_dataset.json

with open('finetune_dataset.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

def convert_to_conversations(item):
    """Convert format CodeBuddy ke format chat Unsloth."""
    return {
        'conversations': [
            {
                'role': 'user',
                'content': f"{item['instruction']}\n\n{item['input']}"
            },
            {
                'role': 'assistant', 
                'content': item['output']
            }
        ]
    }

converted = [convert_to_conversations(item) for item in raw['data']]
dataset = Dataset.from_list(converted)

print(f'Dataset: {len(dataset)} contoh')
print('Contoh pertama:')
print(json.dumps(dataset[0], indent=2, ensure_ascii=False)[:500])

In [ ]:
# Step 5: Apply chat template
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')
dataset = standardize_sharegpt(dataset)

def format_prompts(examples):
    texts = tokenizer.apply_chat_template(
        examples['conversations'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': texts}

dataset = dataset.map(format_prompts, batched=True)
print('Dataset siap. Sample:')
print(dataset[0]['text'][:300])

In [ ]:
# Step 6: Training — dihemat VRAM untuk Colab T4 (~15 GB)
# Penyebab OOM umum: max_seq_length 2048 + batch 2 → logits sangat besar saat backward.
# Solusi: seq 1024, batch 1, accum 8 (effective batch 8), gradient_checkpointing.
import gc
import os

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import torch

gc.collect()
torch.cuda.empty_cache()

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

MAX_LEN = 1024  # jika masih OOM, coba 768

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_LEN,
    dataset_num_proc=1,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="codebuddy-gemma4-checkpoints",
        report_to="none",
    ),
)

print("Mulai training...")
trainer_stats = trainer.train()
print(f"Training selesai! Loss akhir: {trainer_stats.training_loss:.4f}")

In [ ]:
# Step 7: Test model hasil fine-tuning
# Processor Gemma 4 multimodal mengharuskan content = list part (bukan string tunggal).
import torch

FastModel.for_inference(model)

teks_pertanyaan = (
    'Analisis kode Python ini dan jelaskan errornya dalam Bahasa Indonesia\n\nprin("Halo Dunia")'
)
test_input = [
    {
        "role": "user",
        "content": [{"type": "text", "text": teks_pertanyaan}],
    }
]

encoded = tokenizer.apply_chat_template(
    test_input,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)
# Bisa tensor langsung atau BatchEncoding tergantung versi transformers
if isinstance(encoded, torch.Tensor):
    input_ids = encoded.to("cuda")
else:
    input_ids = encoded["input_ids"].to("cuda")

outputs = model.generate(
    input_ids=input_ids,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][input_ids.shape[1] :], skip_special_tokens=True)
print("Respons model fine-tuned:")
print(response)

In [ ]:
# Step 8: Ekspor AMAN — adapter LoRA + checkpoint (ZIP). Tanpa merge GGUF di Colab.
#
# save_pretrained_gguf di Colab gratis sering OOM → kernel restart (tanpa Traceback). Untuk submission:
# unduh ZIP ini + screenshot; GGUF / Ollama bisa dari mesin lokal atau Step 8b (opsional).
import gc
import os
import shutil

os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Jalankan Step 1–7 dulu di sesi yang sama.")

for nama_objek in ("trainer", "trainer_stats"):
    if nama_objek in globals():
        del globals()[nama_objek]
gc.collect()

import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()

nama_folder = "codebuddy_colab_export"
if os.path.isdir(nama_folder):
    shutil.rmtree(nama_folder)
os.makedirs(nama_folder, exist_ok=True)

path_adapter = os.path.join(nama_folder, "lora_adapter")
print("Menyimpan adapter + tokenizer →", path_adapter)
model.save_pretrained(path_adapter)
tokenizer.save_pretrained(path_adapter)

if os.path.isdir("codebuddy-gemma4-checkpoints"):
    shutil.copytree(
        "codebuddy-gemma4-checkpoints",
        os.path.join(nama_folder, "codebuddy-gemma4-checkpoints"),
        dirs_exist_ok=True,
    )
    print("Checkpoint training ikut disalin.")

if os.path.isdir("/content/drive/MyDrive"):
    path_zip = "/content/drive/MyDrive/codebuddy_colab_export"
else:
    path_zip = "/content/codebuddy_colab_export"

shutil.make_archive(path_zip, "zip", root_dir=".", base_dir=nama_folder)
print("ZIP:", path_zip + ".zip")
print("Unduh dari Files / Drive. GGUF: Step 8b (opsional) atau Unsloth docs — Saving to GGUF (lokal).")

## Step 8b — GGUF di Colab (**boleh di-skip sepenuhnya**)

Di **Colab gratis**, `save_pretrained_gguf` **hampir selalu** memicu **OOM → kernel restart** (bukan salah Anda).

**Untuk submission / Unsloth prize:** sudah **cukup** dengan **Step 6 (loss)** + **Step 7 (contoh output)** + **Step 8 (ZIP adapter + checkpoint)** yang sudah Anda unduh. **Tidak wajib** punya GGUF dari Colab.

GGUF untuk Ollama: nanti dari **mesin lokal** atau **Colab Pro / GPU besar** — lihat [Unsloth: Saving to GGUF](https://docs.unsloth.ai/basics/inference-and-deployment/saving-to-gguf).

Cell kode di bawah **default tidak menjalankan** export GGUF. Ubah `JALANKAN_GGUF_DI_COLAB = True` hanya jika Anda yakin VRAM/RAM mencukupi.


In [ ]:
# Step 8b — export GGUF (default OFF: Colab gratis hampir selalu OOM)
JALANKAN_GGUF_DI_COLAB = False  # set True hanya di Colab Pro / GPU besar jika ingin benar-benar coba

import gc
import os

if not JALANKAN_GGUF_DI_COLAB:
    print("Step 8b dilewati (JALANKAN_GGUF_DI_COLAB=False).")
    print("ZIP Step 8 sudah cukup untuk bukti fine-tuning; GGUF → mesin lokal (Unsloth docs).")
else:
    os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("Jalankan Step 2–7 dulu; Step 8 (ZIP) sebaiknya sudah jalan.")
    for nama_objek in ("trainer", "trainer_stats"):
        if nama_objek in globals():
            del globals()[nama_objek]
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    folder_gguf = (
        "/content/drive/MyDrive/codebuddy_gguf"
        if os.path.isdir("/content/drive/MyDrive")
        else "codebuddy_gguf"
    )
    os.makedirs(folder_gguf, exist_ok=True)
    model.save_pretrained_gguf(
        folder_gguf,
        tokenizer,
        quantization_method="fast_quantized",
        maximum_memory_usage=0.28,
    )
    print("GGUF selesai:", folder_gguf)


In [ ]:
# Step 9: Upload ke Hugging Face Hub (opsional)
# push_to_hub_gguf juga berat di Colab — lebih aman: unggah ZIP Step 8 lewat web HF (Files).
# Ganti dengan username HF kamu
HF_USERNAME = 'username-kamu'
HF_TOKEN = 'hf_...'  # dari https://huggingface.co/settings/tokens

model.push_to_hub_gguf(
    f'{HF_USERNAME}/codebuddy-gemma4-indonesian-tutor',
    tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN,
)
print(f'Model tersedia di: https://huggingface.co/{HF_USERNAME}/codebuddy-gemma4-indonesian-tutor')

## Setelah Fine-tuning Selesai

### Gunakan model di Ollama (lokal)

**Jika Anda punya file `.gguf` dari Step 8b (atau konversi lokal):**

```bash
# Buat Modelfile
cat > Modelfile << 'EOF'
FROM ./codebuddy-gemma4-finetuned/unsloth.Q4_K_M.gguf
SYSTEM "Kamu adalah CodeBuddy, tutor Python untuk pelajar Indonesia. Selalu jawab dalam Bahasa Indonesia yang ramah dan menyemangati."
EOF

# Buat model Ollama
ollama create codebuddy -f Modelfile

# Test
ollama run codebuddy "Jelaskan apa itu variabel dalam Python"
```

**Hanya ada ZIP Step 8 (adapter + checkpoint, tanpa GGUF):** buat `.gguf` dulu di mesin dengan RAM cukup mengikuti dokumentasi Unsloth (*Saving to GGUF* / merge lokal). Untuk submission hackathon biasanya **cukup ZIP + screenshot training**; Ollama dengan model custom bisa menyusul.

### Update config CodeBuddy
```python
# utils/config.py
ollama_model: str = "codebuddy"  # pakai model fine-tuned
```